# Experimento 2. Emisión frente a clima

Una concentración baja puede ser menos emisión o más viento. Para separarlas hay dos familias de métodos, y aquí se prueban primero donde se conoce la respuesta y luego con datos reales:

- **Estimador normalizado**: para cada día, NOₓ observado ÷ NOₓ que daría **solo el clima** con una emisión fija (el modelo de transporte con la emisión a priori). Si el modelo captura el clima, ese índice cambia solo cuando cambia la emisión. El cambio es el cociente de los índices de dos tipos de día.
- **`rf_deweather`**: un bosque aleatorio aprende la concentración a partir de la meteorología y el calendario, y luego se lee el efecto del tipo de día con el clima fijo (como en `rmweather`). No usa un modelo de transporte, y da cambios relativos, no toneladas.

Se añaden dos referencias: la **concentración sin corregir el clima** (domingo ÷ día hábil de las medias) y la **inversión** de la línea base (`inversion_plume`). Todas dan el cociente **domingo ÷ día hábil**, el cambio de emisión que se puede verificar con conteos de tráfico (0.65–0.73).

In [ ]:
from nox_ml.data.inputs import load_inputs
from nox_ml.validation import report
from nox_ml.validation import report_experiments as rx

RUNS = report.find_runs()
ROOT = RUNS.parent
inputs = load_inputs(ROOT / "data", "2023-11-01", "2023-12-31")
print(
    f"{len(inputs.hours)} horas de tarde, {inputs.n_station_rows} observaciones de estación, "
    f"{inputs.n_pixels} píxeles de TEMPO (CDMX, noviembre–diciembre de 2023)"
)

## 1. Gemelo con respuesta conocida

Los datos son sintéticos pero usan el clima **real** de las 366 horas: el modelo de la línea base responde a unas emisiones conocidas (fuentes puntuales y regiones viales al azar alrededor de su a priori; sábado = 0.8 y domingo = `s` veces el día hábil), más un fondo horario y ruido de estación y de TEMPO. Se aplican los estimadores y se compara con la respuesta.

**Gemelo A: el domingo emite la mitad** (`s = 0.5`).

In [ ]:
truth = rx.make_truth(inputs, seed=0, saturday=0.8, sunday=0.5)
twin_a = rx.simulate_observations(inputs, truth, seed=0)
table_a = rx.estimator_table(twin_a)
table_a.round(2)

In [ ]:
rx.plot_estimators(table_a, truth=0.5)

**Gemelo B: no cambia nada** (`s = 1`). Cualquier diferencia entre domingo y día hábil es clima.

In [ ]:
truth_b = rx.make_truth(inputs, seed=0, saturday=1.0, sunday=1.0)
twin_b = rx.simulate_observations(inputs, truth_b, seed=0)
table_b = rx.estimator_table(twin_b)
table_b.round(2)

In [ ]:
rx.plot_estimators(table_b, truth=1.0)

**Lectura.**

- **Sin cambio de emisión (B), la concentración sin corregir dice 0.87**: una «baja» de 13 % que es solo clima (el clima de los domingos de este periodo diluyó más). El estimador normalizado da 1.03, `rf_deweather` 0.98 y la inversión 1.00. Es la razón de ser de la normalización.
- **Con la mitad de emisión (A)**, el estimador normalizado da 0.63, **no 0.5**. Subestima el cambio porque el modelo no tiene fondo: el NOₓ observado incluye unos 5 ppb que no vienen de la ciudad, y eso acerca el cociente a 1. La investigación ya lo había visto («ligera subestimación»). El bosque da 0.54 (0.45–0.63) y la inversión 0.50.
- La inversión sale casi exacta porque **el gemelo se generó con su propio modelo**: aquí comprueba el código, no el método. Con datos reales no tiene esa ventaja (siguiente sección).
- El intervalo del normalizado en A (0.59–0.67) **no contiene** la respuesta: el bootstrap mide el ruido, no el sesgo del método. Por eso hace falta el placebo.

## 2. Noviembre–diciembre de 2023, datos reales

Los mismos estimadores, ahora sobre las observaciones de las estaciones. La franja es lo que implican los conteos de tráfico de Infovial (domingo ÷ hábil de 0.65 a 0.73), que es una referencia aproximada y no la verdad.

In [ ]:
table_real = rx.estimator_table(inputs)
table_real.round(2)

In [ ]:
rx.plot_estimators(table_real, band=rx.TRAFFIC_BAND)

**Lectura.**

- Corregir el clima **sube** el cociente de 0.56 a 0.66: el clima hizo que los domingos parecieran más limpios de lo que su emisión explica, como en el gemelo B. El estimador normalizado cae en la franja del tráfico.
- Los métodos no coinciden: `rf_deweather` da 0.53 (0.42–0.57) y la inversión 0.42 (0.35–0.54). Según el gemelo A, el normalizado tiende a quedar por encima de la respuesta y el bosque queda cerca. Con estos datos no hay forma de saber cuál tiene razón; lo que sí se puede decir es que **todos dan un domingo claramente por debajo del día hábil** y que el tamaño de la baja depende del método (0.42–0.66).
- Solo la inversión y el normalizado usan un modelo de transporte; el bosque no, y aun así queda entre ambos.

## 3. El placebo

Un estimador siempre da un número. El placebo responde: **¿qué número daría si no hubiera ningún cambio?** Se toman solo días hábiles, se eligen al azar tantos como domingos tiene el periodo (11 con los festivos), se les llama «domingos falsos» y se calcula el cociente. Como entre ellos no hay cambio de emisión, el resultado debería estar en 1; su dispersión es el ruido de un resultado nulo. Se repite 2,000 veces con el estimador normalizado, y 8 veces con el bosque (cada repetición lo vuelve a ajustar). El centro del placebo de un modelo debe estar en 1 ± 0.15 para que el modelo compita (`eligible`).

Primero, de dónde sale el ruido: el índice de cada día.

In [ ]:
daily = rx.daily_normalized(inputs)
rx.plot_obs_vs_model(daily)

In [ ]:
rx.plot_daily_index(daily)

**Lectura.** En el primer gráfico, cada tipo de día queda sobre una recta que sale del origen y cuya pendiente es su índice de emisión: la de los domingos es más baja que la de los días hábiles. Pero los puntos se dispersan mucho alrededor de cada recta: dos días hábiles estancados (a la derecha) quedan muy por debajo de su recta, porque el modelo sobrestima lo que daría el clima. En el segundo, entre días hábiles el índice varía alrededor de un tercio (35 %) **sin que cambie ninguna emisión**. Con 11 domingos contra 41 días hábiles, esa variación decide qué tan pequeño puede ser un cambio detectable.

In [ ]:
daily_ratio = rx.index_ratio(daily)
null = rx.fake_sunday_ratios(daily)
model_null = rx.model_placebo_ratios(inputs, "rf_deweather")
rx.placebo_summary(null, model_null).round(2)

In [ ]:
rx.plot_placebo(null, daily_ratio, model_null)

**Lectura.**

- **Normalizado.** Con cero cambio, el 90 % de los cocientes cae entre 0.78 y 1.29: un semiancho de 0.25, es decir, **solo se detecta un cambio de más o menos 25 %**. El domingo real, 0.66 (0.55–0.77), queda justo por debajo de ese límite: es un cambio detectable, pero al borde y con un intervalo ancho.
- **Bosque.** Su placebo está centrado (0.97) y es más estrecho (semiancho de 0.09), así que pasa la compuerta de 1 ± 0.15, y su domingo (0.53) queda lejos de la nube: es la evidencia más fuerte de que el domingo emite menos. Con solo 8 repeticiones el ancho es una estimación gruesa.
- Un resultado que cae dentro de la nube gris **no se puede distinguir de un cambio nulo**: es lo que pasa con cambios de menos de 25 % medidos con el estimador normalizado de estaciones.